# 0 Instalar Dependencias

In [14]:
!pip install langchain langchain-openai langchain-community

In [15]:
from dotenv import load_dotenv
load_dotenv()



True

In [20]:
import os
os.environ.get("CHAVE_API_OPENAI")
os.environ.get("base_url")
os.environ.get("api_key")

In [31]:
import getpass
api_key = os.getenv("CHAVE_API_OPENAI")
if not api_key:
    api_key = getpass.getpass()


base_url = os.environ.get("base_url")
api_key_local = os.environ.get("api_key")

In [25]:
from langchain_community.document_loaders import TextLoader

documento = TextLoader("documentos/GTB_gold_Nov23.txt", encoding="utf-8").load()


In [26]:
documento

[Document(metadata={'source': 'documentos/GTB_gold_Nov23.txt'}, page_content='\n1\n1 \nVersão: novembro 2023 \n2021 \n \n \n \nPrograma de Cartão da Edição Mastercard Gold  \nGuia de Benefícios \n Informações importantes. Leia e guarde as informações. \n \nEste Guia de Benefícios contém informações detalhadas sobre serviços abrangentes de viagem, seguros \ne assistência aos quais você terá acesso como portador de cartão preferencial. Esses benefícios e serviços \nestão em vigor para portadores do cartão Mastercard Gold elegível a partir de 1 de Novembro de  2023. \nEste Guia substitui qualquer  guia ou comunicação de programa que você recebeu anteriormente. \n \nAs informações contidas neste documento são apresentadas somente com propósito informativo. Não \npretendem  ser  uma  descrição  completa  de  todos  os  termos,  condições,  limitações, exclusões  ou  outras \ndisposições  de  qualquer  programa  ou  benefícios  de  seguro  fornecidos  por,  para,  ou  emitidos  para  a \nMas

# Chunking


In [27]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000, chunk_overlap=100
)

pedacos = splitter.split_documents(documento)

In [28]:
len(pedacos)

5

In [29]:
pedacos[2]

Document(metadata={'source': 'documentos/GTB_gold_Nov23.txt'}, page_content='uma perda ou pedido de serviços forem efetuados, se você intencionalmente ocultar ou fizer \ninterpretação  errônea  de  qualquer  fato  material  ou  circunstância,  ou  fornecer  informação  fraudulenta \nrelativa  aos  planos  de  seguro  ou  outros  serviços  aqui  descritos  para:  A  Mastercard  International,  a \nEmpresa de Seguros, a instituição financeira que emitiu a Conta do cartão ou qualquer outra empresa \nque estiver prestando serviços e/ou administração em nome destes programas. \n \nPara  dar  entrada  em  uma  ocorrência/sinistro  ou  para  obter  mais  informações  sobre  qualquer um \ndesses serviços, ligue para o número gratuito do Mastercard Global Service™ específico para o seu país, \nou ligue a cobrar para os Estados Unidos no número 1-636-722-8881 (Português). \n “cartão” refere-se ao cartão Mastercard Gold. \n \n“portador de cartão”, “você”, e “seu” referem-se a um  portador do cart

In [34]:
from langchain_openai import OpenAIEmbeddings

embeddings_model = OpenAIEmbeddings(api_key=api_key)

In [35]:
embeddings_model.model

'text-embedding-ada-002'

In [36]:
pedacos[0].page_content

'1\n1 \nVersão: novembro 2023 \n2021 \n \n \n \nPrograma de Cartão da Edição Mastercard Gold  \nGuia de Benefícios \n Informações importantes. Leia e guarde as informações. \n \nEste Guia de Benefícios contém informações detalhadas sobre serviços abrangentes de viagem, seguros \ne assistência aos quais você terá acesso como portador de cartão preferencial. Esses benefícios e serviços \nestão em vigor para portadores do cartão Mastercard Gold elegível a partir de 1 de Novembro de  2023. \nEste Guia substitui qualquer  guia ou comunicação de programa que você recebeu anteriormente. \n \nAs informações contidas neste documento são apresentadas somente com propósito informativo. Não \npretendem  ser  uma  descrição  completa  de  todos  os  termos,  condições,  limitações, exclusões  ou  outras \ndisposições  de  qualquer  programa  ou  benefícios  de  seguro  fornecidos  por,  para,  ou  emitidos  para  a \nMastercard. \n \nNome  do Representante: MASTERCARD DO BRASIL LTDA.  CNPJ 01.248.2

In [38]:
embeddings_model.embed_query(pedacos[0].page_content) #nao pago o openAI api's :\

RateLimitError: Error code: 429 - {'error': {'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, read the docs: https://platform.openai.com/docs/guides/error-codes/api-errors.', 'type': 'insufficient_quota', 'param': None, 'code': 'insufficient_quota'}}

In [43]:
from langchain_community.vectorstores import InMemoryVectorStore

vectorstore = InMemoryVectorStore.from_documents(
    documents=pedacos, embedding=embeddings_model
)

RateLimitError: Error code: 429 - {'error': {'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, read the docs: https://platform.openai.com/docs/guides/error-codes/api-errors.', 'type': 'insufficient_quota', 'param': None, 'code': 'insufficient_quota'}}

In [42]:
retriever = vectorstore.as_retriever(search_kwargs={"k", 2})

NameError: name 'vectorstore' is not defined

In [44]:
retriever.invoke("Seguro viagem")

NameError: name 'retriever' is not defined

In [46]:
query = "Como eu devo proceder caso tenha um item comprado roubado?"

query_embed = embeddings_model.embed_query(query)
query_embed

RateLimitError: Error code: 429 - {'error': {'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, read the docs: https://platform.openai.com/docs/guides/error-codes/api-errors.', 'type': 'insufficient_quota', 'param': None, 'code': 'insufficient_quota'}}

In [ ]:
simular_chunks = retriever.invoke(query)


NameError: name 'retriever' is not defined

In [ ]:
similar_texts = [pedaco.pege_content for pedaco in similar_texts]

In [48]:
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_messages(
   [ ("system", "responda usando exclusivamente os conteudos fornecidos. \n\Contexto:\n{contexto}"),
    ("human", "{query}")
])

<>:4: SyntaxWarning: invalid escape sequence '\C'
<>:4: SyntaxWarning: invalid escape sequence '\C'
/var/folders/gx/9vrjjlfd295c01q_1m9vd9kc0000gn/T/ipykernel_45595/1961359806.py:4: SyntaxWarning: invalid escape sequence '\C'
  [ ("system", "responda usando exclusivamente os conteudos fornecidos. \n\Contexto:\n{contexto}"),


In [49]:
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser

modelo = ChatOpenAI(
    model='gpt-4.1-nano',
    temperature=0.2,
    api_key=api_key
)

In [ ]:
modelo.invoke(query)

In [ ]:
cadeia = prompt | modelo | StrOutputParser()

trechos = retriever.invoke(query)
contexto = "\n\n".join(trecho.page_content for trecho in trechos)

cadeia.invoke({"query": query, "contexto": contexto})